In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%pip install -U scikit-learn

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Environment ready!")
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", __import__("sklearn").__version__)


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp313-cp313-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.1-cp313-cp313-win_amd64.whl (8.2 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---

In [8]:
!pip install pdfplumber

   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
           ~~~~~~~~~~~~~^^^^^
  File "C:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
           ~~~~~~~~~~~~~^^^^^
  File "C:\Users\Asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 100, in read
    data: bytes = self.__fp.read(amt)
                  ~~~~~~~~~~~~~~^^^^

In [9]:
# ==========================================
# STUDENT PERFORMANCE - DECISION TREE
# ==========================================

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# ==========================================
# 1. Load data from PDF
# ==========================================

# Install pdfplumber first if required:
# !pip install pdfplumber

import pdfplumber

pdf_file = "student_performance.pdf"

all_tables = []

with pdfplumber.open(pdf_file) as pdf:

    for page in pdf.pages:
        tables = page.extract_tables()

        for table in tables:
            if table:
                all_tables.append(table)

print("Number of tables found:", len(all_tables))


# ==========================================
# 2. Convert table to DataFrame
# ==========================================

if len(all_tables) == 0:
    print("No table was found in the PDF.")
else:

    # Take the first table
    data = all_tables[0]

    # First row as column names
    df = pd.DataFrame(data[1:], columns=data[0])

    # Remove empty columns
    df = df.dropna(axis=1, how="all")

    print("\nDataset:")
    print(df.head())

    print("\nColumns:")
    print(df.columns.tolist())


# ==========================================
# 3. Clean column names
# ==========================================

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("\nCleaned columns:")
print(df.columns.tolist())


# ==========================================
# 4. Convert numerical columns
# ==========================================

possible_numeric_columns = [
    "age",
    "studytime",
    "failures",
    "assignments",
    "attendance",
    "result"
]

for column in possible_numeric_columns:

    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )


# Remove rows containing missing values
df = df.dropna()

print("\nClean dataset:")
print(df.head())

print("\nDataset shape:")
print(df.shape)


# ==========================================
# 5. Basic information
# ==========================================

print("\nDataset information:")
print(df.info())

print("\nStatistical summary:")
print(df.describe())


# ==========================================
# 6. Group data
# ==========================================

if "result" in df.columns:

    print("\nAverage study time and failures by result:")

    print(
        df.groupby("result")[["studytime", "failures"]].mean()
    )


# ==========================================
# 7. Scatter plot
# ==========================================

if all(
    column in df.columns
    for column in ["studytime", "failures", "result"]
):

    plt.figure(figsize=(7, 4))

    plt.scatter(
        df["studytime"],
        df["failures"],
        c=df["result"]
    )

    plt.xlabel("Study Time")
    plt.ylabel("Failures")
    plt.title("Student Performance Pattern")

    plt.show()


# ==========================================
# 8. Features and target
# ==========================================

X = df[["studytime", "failures"]]

y = df["result"]

print("\nX shape:", X.shape)
print("y shape:", y.shape)


# ==========================================
# 9. Check target values
# ==========================================

print("\nResult distribution:")
print(y.value_counts())


# ==========================================
# 10. Split dataset
# ==========================================

# Check whether stratification is possible

if y.value_counts().min() >= 2:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

else:

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )


print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))


# ==========================================
# 11. Create Decision Tree
# ==========================================

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)


# ==========================================
# 12. Train model
# ==========================================

model.fit(X_train, y_train)

print("\nModel trained!")


# ==========================================
# 13. Prediction
# ==========================================

y_pred = model.predict(X_test)

print("\nPredictions:")
print(y_pred)


# ==========================================
# 14. Accuracy
# ==========================================

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:")
print(accuracy)


# ==========================================
# 15. Confusion Matrix
# ==========================================

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


# ==========================================
# 16. Classification Report
# ==========================================

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred
    )
)


# ==========================================
# 17. Display Decision Tree
# ==========================================

plt.figure(figsize=(12, 7))

plot_tree(
    model,
    feature_names=[
        "studytime",
        "failures"
    ],
    class_names=[
        str(value)
        for value in model.classes_
    ],
    filled=True
)

plt.title("Student Performance Decision Tree")

plt.show()

ModuleNotFoundError: No module named 'pdfplumber'